# Kafka Quickstart with Docker Compose

This notebook shows how to:

1. Start Kafka using the `docker/docker-compose.yml` in this repo
2. Verify Kafka is running
3. Produce and consume a few test messages from Python

> **Prerequisites**
>
> - Docker Desktop installed and running
> - This project checked out locally
> - Python environment with `kafka-python` installed


## Kafka 101: What, Why, and How

**Apache Kafka** is a distributed event streaming platform. It acts as a durable, scalable log of events that many producers can write to and many consumers can read from.

### Core concepts
- **Broker**: A single Kafka server. A Kafka cluster is made of multiple brokers.
- **Topic**: A named stream of events (e.g. `orders`, `user-signups`). Producers write to topics; consumers read from them.
- **Partition**: Each topic is split into partitions (append-only logs) for scalability and parallelism. Ordering is guaranteed **within a partition**.
- **Offset**: A monotonically increasing number indicating a message's position in a partition.
- **Producer**: An application that writes messages to topics.
- **Consumer**: An application that reads messages from topics.
- **Consumer group**: A group of consumers that share work. Each partition is processed by at most one consumer in the group.

### How Kafka works (high level)
1. **Producers append messages** to topic partitions.
2. **Brokers store data** on disk and replicate it to other brokers for fault tolerance.
3. **Consumers poll for new messages** from their assigned partitions.
4. **Offsets** track how far each consumer has read; they can be committed so consumers can resume from where they left off.
5. **Retention** keeps messages for a configured time/size, allowing replay of history.

### Why and when to use Kafka
- **Decoupling services**: Producers don't need to know who consumes their events.
- **Real-time data pipelines**: Build streaming ETL from apps into data lakes/warehouses.
- **Event-driven architectures**: Services react to business events ("OrderPlaced", "UserSignedUp", etc.).
- **Log aggregation and observability**: Centralize logs, metrics, and clickstreams for analysis.

Kafka is a great fit when you need high-throughput, replayable event streams with multiple independent consumers. For small, simple job queues, a lighter-weight system may be enough.


> **Tip**: The ingestion scripts (`src/ingestion/producer.py` and `bronze_consumer.py`) use the same Kafka settings defined in `src/config.py`, so once you can produce/consume from this notebook, the CLI tools will work too.

In [ ]:
# Install kafka-python (run once per environment)
!pip install kafka-python


## 1. Start Kafka with Docker Compose

From a terminal in the project root, run:

```bash
docker compose -f docker/docker-compose.yml up -d kafka
```

Wait until logs show `Kafka Server started` (you can check with `docker compose -f docker/docker-compose.yml logs kafka`).


In [7]:
!docker compose -f docker/docker-compose.yml up -d kafka

 Network docker_app-tier Creating 
 Network docker_app-tier Created 
 Container docker-kafka-1 Creating 
 Container docker-kafka-1 Created 
 Container docker-kafka-1 Starting 
 Container docker-kafka-1 Started 


### Understanding the Kafka configuration in `docker-compose.yml`

In `docker/docker-compose.yml`, the `kafka` service has several important environment variables:

- **`KAFKA_NODE_ID=0`**: Unique ID of this Kafka node in the KRaft cluster. Since we run a single broker, we just use `0`.
- **`KAFKA_PROCESS_ROLES=broker,controller`**: Runs this node as both a **broker** (handles client traffic/data) and **controller** (manages cluster metadata). This is common for small dev setups.
- **`KAFKA_LISTENERS=PLAINTEXT://:9092,CONTROLLER://:9093`**: Declares two listeners:
  - `PLAINTEXT://:9092` for client connections (producers/consumers)
  - `CONTROLLER://:9093` for internal controller traffic
- **`KAFKA_ADVERTISED_LISTENERS=PLAINTEXT://localhost:9092`**: The address the broker "advertises" to clients. From your host (this notebook), you connect to `localhost:9092`, so the broker tells clients to use that.
- **`KAFKA_LISTENER_SECURITY_PROTOCOL_MAP=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT`**: Maps each listener name (`CONTROLLER`, `PLAINTEXT`) to the `PLAINTEXT` protocol (no TLS/auth), which is fine for local dev.
- **`KAFKA_CONTROLLER_LISTENER_NAMES=CONTROLLER`**: Tells Kafka which listener is used for the controller role (`CONTROLLER` on port 9093).
- **`KAFKA_INTER_BROKER_LISTENER_NAME=PLAINTEXT`**: Chooses which listener brokers use to talk to each other. Here they would use the `PLAINTEXT` listener on port 9092.
- **`KAFKA_CONTROLLER_QUORUM_VOTERS=0@kafka:9093`**: Defines the KRaft quorum. In a multi-broker cluster this would list all controllers; here we only have one: node `0` at `kafka:9093` (the service name and controller port).
- **`KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR=1`**: Stores the internal `__consumer_offsets` topic with a replication factor of 1 (only this broker). Required because we have only one broker.
- **`KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR=1`** and **`KAFKA_TRANSACTION_STATE_LOG_MIN_ISR=1`**: Similar single-broker-safe settings for Kafka's internal transactional state topics.

Together, these settings configure a **single-broker, single-controller KRaft cluster** that works well for local development and can be reached from this notebook at `localhost:9092`. 

In [4]:
from kafka import KafkaProducer, KafkaConsumer
from kafka.errors import NoBrokersAvailable
import time

bootstrap_servers = ["localhost:9092"]  # use "kafka:9092" from inside another container
topic = "demo-topic"

# Wait for broker to be ready
for i in range(10):
    try:
        producer = KafkaProducer(bootstrap_servers=bootstrap_servers)
        break
    except NoBrokersAvailable:
        print("Broker not ready yet, retrying...")
        time.sleep(2)
else:
    raise RuntimeError("Kafka broker not reachable on localhost:9092")

print("Producer connected. Sending messages...")
for i in range(10):
    producer.send(topic, f"message-{i}".encode("utf-8"))
producer.flush()
producer.close()
print("Messages produced.")


Producer connected. Sending messages...
Messages produced.


In [5]:
from kafka import KafkaConsumer

bootstrap_servers = ["localhost:9092"]  # or ["kafka:9092"] if running inside Docker
topic = "demo-topic"

consumer = KafkaConsumer(
    topic,
    bootstrap_servers=bootstrap_servers,
    group_id="demo-group",          # <-- fixed group id
    auto_offset_reset="latest",     # only used if this group has no offsets yet
    enable_auto_commit=True,        # commit offsets so next run resumes
    consumer_timeout_ms=5000,       # stop iteration after 5s idle
)

print("Consuming messages from", topic)
count = 0
for msg in consumer:
    print(f"Received: {msg.value.decode('utf-8')}")
    count += 1

consumer.close()

if count == 0:
    print("No messages received. Make sure you re‑ran the producer cell and it reported 'Messages produced.'")


Consuming messages from demo-topic
Received: message-0
Received: message-1
Received: message-2
Received: message-3
Received: message-4
Received: message-5
Received: message-6
Received: message-7
Received: message-8
Received: message-9


## Producing to a Cluster with Multiple Brokers

In a real Kafka cluster you often have multiple brokers (for example, 3). Clients don't connect to just one broker; instead they are given a **bootstrap list** of brokers. The client will:

- Use any available broker from the list to discover the full cluster metadata.
- Automatically handle broker failures as long as at least one broker from the list is reachable.

Below is an example of how you would configure a producer for a cluster with three brokers. (In this tutorial we only run one broker, but the code pattern is the same.)


## Using the IMDb producer and bronze consumer from this project

Once Kafka is running, you can stream real IMDb Movie Reviews into Kafka and land them in the bronze layer.

From a terminal in the project root:

```bash
# Stream 100 IMDb reviews to the `imdb-reviews` topic
python -m src.ingestion.producer --mode batch --limit 100

# In another terminal, consume them into the bronze layer (MinIO/S3)
python -m src.ingestion.bronze_consumer --batch-size 50
```

The bronze consumer will write newline-delimited JSON files under the `bronze/imdb/` prefix in your configured S3/MinIO bucket (see `src/config.py`).

You can then point downstream transformation jobs (e.g. `silver_job.py`) at the bronze IMDb data to clean, enrich, and train models on top of the ingested reviews.

### How to configure multiple brokers (a small cluster)

So far our `docker-compose.yml` runs a **single broker**. In a real setup you often run **3 brokers** for fault tolerance. At a high level, configuring multiple brokers means:

1. **One service per broker** in `docker-compose.yml` (e.g. `kafka-1`, `kafka-2`, `kafka-3`).
2. Each broker gets a unique **`KAFKA_NODE_ID`** (1, 2, 3, ...).
3. Each broker has its own **data volume** (e.g. `kafka_data_1`, `kafka_data_2`, `kafka_data_3`).
4. All brokers share the same **controller quorum** via `KAFKA_CONTROLLER_QUORUM_VOTERS`.
5. Internal topics use a replication factor equal to the number of brokers.

A simplified example (not wired into this repo, just for reference) would look like this:

```yaml
services:
  kafka-1:
    image: apache/kafka:latest
    environment:
      - KAFKA_NODE_ID=1
      - KAFKA_PROCESS_ROLES=broker,controller
      - KAFKA_LISTENERS=PLAINTEXT://:9092,CONTROLLER://:9093
      - KAFKA_ADVERTISED_LISTENERS=PLAINTEXT://kafka-1:9092
      - KAFKA_CONTROLLER_LISTENER_NAMES=CONTROLLER
      - KAFKA_LISTENER_SECURITY_PROTOCOL_MAP=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      - KAFKA_CONTROLLER_QUORUM_VOTERS=1@kafka-1:9093,2@kafka-2:9093,3@kafka-3:9093
      - KAFKA_OFFSETS_TOPIC_REPLICATION_FACTOR=3
      - KAFKA_TRANSACTION_STATE_LOG_REPLICATION_FACTOR=3
      - KAFKA_TRANSACTION_STATE_LOG_MIN_ISR=2
    volumes:
      - kafka_data_1:/var/lib/kafka

  kafka-2:
    image: apache/kafka:latest
    environment:
      - KAFKA_NODE_ID=2
      - KAFKA_PROCESS_ROLES=broker,controller
      - KAFKA_LISTENERS=PLAINTEXT://:9092,CONTROLLER://:9093
      - KAFKA_ADVERTISED_LISTENERS=PLAINTEXT://kafka-2:9092
      - KAFKA_CONTROLLER_LISTENER_NAMES=CONTROLLER
      - KAFKA_LISTENER_SECURITY_PROTOCOL_MAP=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      - KAFKA_CONTROLLER_QUORUM_VOTERS=1@kafka-1:9093,2@kafka-2:9093,3@kafka-3:9093
    volumes:
      - kafka_data_2:/var/lib/kafka

  kafka-3:
    image: apache/kafka:latest
    environment:
      - KAFKA_NODE_ID=3
      - KAFKA_PROCESS_ROLES=broker,controller
      - KAFKA_LISTENERS=PLAINTEXT://:9092,CONTROLLER://:9093
      - KAFKA_ADVERTISED_LISTENERS=PLAINTEXT://kafka-3:9092
      - KAFKA_CONTROLLER_LISTENER_NAMES=CONTROLLER
      - KAFKA_LISTENER_SECURITY_PROTOCOL_MAP=CONTROLLER:PLAINTEXT,PLAINTEXT:PLAINTEXT
      - KAFKA_CONTROLLER_QUORUM_VOTERS=1@kafka-1:9093,2@kafka-2:9093,3@kafka-3:9093
    volumes:
      - kafka_data_3:/var/lib/kafka

volumes:
  kafka_data_1:
  kafka_data_2:
  kafka_data_3:
```

In this kind of cluster, your client `bootstrap_servers` list would include **all three brokers**, for example:

```python
bootstrap_servers = ["kafka-1:9092", "kafka-2:9092", "kafka-3:9092"]
```

For this tutorial we keep a single-broker setup (simpler to run locally), but the concepts above are what you would use to scale out to a real multi-broker cluster.

## 2. Cleaning up

When you're done experimenting, you can stop Kafka (and other services) with:

```bash
docker compose -f docker/docker-compose.yml down
```


In [8]:
!docker compose -f docker/docker-compose.yml down

 Container docker-kafka-1 Stopping 
 Container docker-kafka-1 Stopped 
 Container docker-kafka-1 Removing 
 Container docker-kafka-1 Removed 
 Network docker_app-tier Removing 
 Network docker_app-tier Removed 
